# High-Accuracy YOLOv8 Training (1k Images / 10k Objects)

Since you have a dense dataset, we are using a Small model with heavy augmentation to ensure the model learns to distinguish individual objects accurately.

## 1. Install Dependencies

In [ ]:
!pip install ultralytics roboflow

## 2. Download Dataset

In [ ]:
from roboflow import Roboflow
import os

rf = Roboflow(api_key="ipRXafHM2fxthdHpXUSC")
project = rf.workspace().project("ako_buto")
dataset = project.version(4).download("yolov8")

## 3. Train with Optimized Settings
We are using **YOLOv8s** (Small) instead of Nano for better feature extraction.

In [ ]:
from ultralytics import YOLO

# Using 's' (small) for significantly better accuracy with dense objects
model = YOLO('yolov8s.pt')

results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=150,
    imgsz=640,
    batch=16,
    patience=100,
    optimizer='AdamW',
    lr0=0.01,
    mosaic=1.0,
    mixup=0.1,
    save=True,
    device=0
)

## 4. Visualize Performance Metrics
This will display the confusion matrix and training curves to help you understand where the model is performing well or poorly.

In [ ]:
from IPython.display import Image, display
import os

results_dir = 'runs/detect/train'

print("--- Training Results Plot ---")
if os.path.exists(f'{results_dir}/results.png'):
    display(Image(filename=f'{results_dir}/results.png', width=1000))

print("\n--- Confusion Matrix ---")
if os.path.exists(f'{results_dir}/confusion_matrix.png'):
    display(Image(filename=f'{results_dir}/confusion_matrix.png', width=800))

print("\n--- F1-Confidence Curve ---")
if os.path.exists(f'{results_dir}/F1_curve.png'):
    display(Image(filename=f'{results_dir}/F1_curve.png', width=800))

print("\n--- Sample Validation Predictions ---")
if os.path.exists(f'{results_dir}/val_batch0_pred.jpg'):
    display(Image(filename=f'{results_dir}/val_batch0_pred.jpg', width=1000))

## 5. Export to ONNX

In [ ]:
# Load the best model from training
best_model = YOLO('runs/detect/train/weights/best.pt')

# Export with dynamic=True for flexibility
path = best_model.export(format="onnx", dynamic=True)
print(f"Exported to: {path}")

## 5. Download Weights

In [ ]:
from google.colab import files
files.download(path)